In [1]:
import pandas as pd
import requests
from io import StringIO
import numpy as np
import ast
import time
import os

In [2]:
def inspect_comptage_multimodal(
    year_start=2021,
    year_end=2027,
    sample_limit=200000,
    timezone="UTC",
):

    dataset_id = "comptage-multimodal-comptages"
    base = f"https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/{dataset_id}"

    # Imprimer les 10 premières lignes
    r = requests.get(f"{base}/records", params={"limit": 10, "timezone": timezone}, timeout=60)
    r.raise_for_status()
    js = r.json()
    df10 = pd.DataFrame(js.get("results", []))
    print("Top 10")
    print(df10.head(10))
    print("\nCoté en bourse:", list(df10.columns))

    # nombre de distinct id_site
    r = requests.get(
        f"{base}/records",
        params={"select": "count(distinct id_site) as n_sites", "limit": 1},
        timeout=60,
    )
    r.raise_for_status()
    n_sites = (r.json().get("results") or [{}])[0].get("n_sites")
    print("\nNombre total de sites")
    print(n_sites)

    # les modes de transport
    r = requests.get(
        f"{base}/records",
        params={
            "select": "mode, count(*) as n",
            "group_by": "mode",
            "order_by": "n desc",
            "limit": 200,
        },
        timeout=60,
    )
    r.raise_for_status()
    df_modes = pd.DataFrame(r.json().get("results", []))
    print(df_modes)

    # ODSQL where：Appuyez sur t pour filtrer vers [year_start, year_end]
    # Utiliser le point de terminaison exports/csv
    start_iso = f"{year_start}-01-01T00:00:00Z"
    end_iso = f"{year_end+1}-01-01T00:00:00Z"
    where = f"t >= '{start_iso}' AND t < '{end_iso}'"

    r = requests.get(
        f"{base}/exports/csv",
        params={
            "where": where,
            "limit": sample_limit,
            "timezone": timezone,
            "delimiter": ",", 
            "with_bom": "false",
        },
        timeout=300,
    )
    r.raise_for_status()
    
    df = pd.read_csv(StringIO(r.text))
    print("10 premières lignes de l'échantillon")
    print(df.head(10))

    # Statistiques sur les valeurs nulles
    na = df.isna().sum().sort_values(ascending=False)
    print("\nNombre de valeurs vides dans chaque colonne")
    print(na[na > 0] if (na > 0).any() else "pas de valeurs vides dans l'échantillon")

    if (df.isna().any(axis=1)).any():
        print("\nLignes contenant des valeurs nulles")
        print(df[df.isna().any(axis=1)].head(10))

    # Heure unifiée
    if "t" not in df.columns:
        print("\nLa colonne t est introuvable dans l'échantillon")
        return

    df["t_utc"] = pd.to_datetime(df["t"], errors="coerce", utc=True)
    bad_t = df["t_utc"].isna().sum()
    if bad_t:
        print(f"\nNombre de lignes où l'analyse a échoué = {bad_t}")

    # Nécessite des caractéristiques parisiennes locales
    df["t_paris"] = df["t_utc"].dt.tz_convert("Europe/Paris")

    print("\nPériode")
    print("t_utc  :", df["t_utc"].min(), "->", df["t_utc"].max())
    print("t_paris:", df["t_paris"].min(), "->", df["t_paris"].max())

    # Intervalle d'enregistrement
    group_cols = [c for c in ["id_site", "mode", "id_trajectoire", "trajectoire"] if c in df.columns]
    if not group_cols:
        print("\nRemarque : colonne de regroupement manquante")
        s = df.sort_values("t_utc")["t_utc"].diff().dropna()
    else:
        dfx = df.dropna(subset=["t_utc"]).sort_values(group_cols + ["t_utc"])
        s = dfx.groupby(group_cols)["t_utc"].diff().dropna()

    delta_h = (s.dt.total_seconds() / 3600.0).round(6)
    vc = delta_h.value_counts().head(20)
    print("\nDes distributions d'intervalles")
    print(vc)

    if len(vc):
        print("\nIntervalle de temps le plus courant:", vc.index[0])

In [3]:
inspect_comptage_multimodal(year_start=2021, year_end=2027, sample_limit=200000, timezone="UTC")

Top 10
  id_trajectoire id_site                    label                          t  \
0   10004_1 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T04:00:00+00:00   
1   10004_1 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T07:00:00+00:00   
2   10004_1 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T09:00:00+00:00   
3   10004_2 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T02:00:00+00:00   
4   10004_2 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T09:00:00+00:00   
5   10004_2 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T19:00:00+00:00   
6   10004_2 -> 1   10004  CF0256_88 rue de Rivoli  2025-12-03T20:00:00+00:00   
7   10004_4 -> 2   10004  CF0256_88 rue de Rivoli  2025-12-03T02:00:00+00:00   
8   10004_4 -> 2   10004  CF0256_88 rue de Rivoli  2025-12-03T05:00:00+00:00   
9   10004_4 -> 2   10004  CF0256_88 rue de Rivoli  2025-12-03T06:00:00+00:00   

                      mode  nb_usagers            voie sens trajectoire  \
0                    Vélos          1

/var/folders/7z/6tydff_d7yx77mr69hghcts80000gn/T/ipykernel_3621/3403547439.py:65: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO(r.text))


10 premières lignes de l'échantillon
  id_trajectoire id_site                                  label  \
0   10022_3 -> 2   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
1   10022_3 -> 2   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
2   10022_3 -> 2   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
3   10022_3 -> 2   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
4   10022_3 -> 2   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
5   10022_3 -> 2   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
6   10022_4 -> 3   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
7   10022_4 -> 3   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
8   10022_4 -> 3   10022  CF0892_65 rue de Rivoli (Bourdonnais)   
9   10022_4 -> 3   10022  CF0892_65 rue de Rivoli (Bourdonnais)   

                           t                     mode  nb_usagers  \
0  2025-12-11T00:00:00+00:00       2 roues motorisées           6   
1  2025-12-11T04:00:00+00:00  Véhicules lourds > 3,5t          45   
2  2025-12-11T06:0

In [4]:
def build_traffic_weather_dataset(
    year_start=2021,
    year_end=2026,
    n_sites=10,
    modes_keep=None,
    keep_trajectoire=False,
    timezone="UTC",
    weather_timezone="UTC",
    chunk_freq="MS",
    out_parquet=None,
    weather_chunk="YS", # Météo par année
    weather_max_retries=8,
):

    if modes_keep is None:
        modes_keep = [
            "Trottinettes + vélos",
            "2 roues motorisées",
            "Trottinettes",
            "Véhicules légers < 3,5t",
            "Véhicules lourds > 3,5t",
        ]

    counts_id = "comptage-multimodal-comptages"
    sites_id = "comptage-multimodal-sites-et-trajectoires-de-comptage"
    base_counts = f"https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/{counts_id}"
    base_sites = f"https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/{sites_id}"

    start_iso = f"{year_start}-01-01T00:00:00Z"
    end_iso = f"{year_end+1}-01-01T00:00:00Z"
    where_time = f"t >= '{start_iso}' AND t < '{end_iso}'"
    where_modes = "mode in (" + ",".join([f"'{m}'" for m in modes_keep]) + ")"
    where_base = f"{where_time} AND {where_modes}"

    # Candidats à la sélection du site
    n_candidates = max(n_sites * 8, n_sites)
    r = requests.get(
        f"{base_counts}/records",
        params={
            "select": "id_site, count(distinct t) as hours_present",
            "where": where_base,
            "group_by": "id_site",
            "order_by": "hours_present desc",
            "limit": n_candidates,
            "timezone": timezone,
        },
        timeout=120,
    )
    r.raise_for_status()
    top_sites_df = pd.DataFrame(r.json().get("results", []))
    if top_sites_df.empty:
        raise RuntimeError("Top sites est vide")

    cand_ids = top_sites_df["id_site"].astype(str).tolist()
    where_cand = "id_site in (" + ",".join([f"'{sid}'" for sid in cand_ids]) + ")"

    # Extraire les coordonnées uniques des sites à partir de l'ensemble de données
    r = requests.get(
        f"{base_sites}/exports/csv",
        params={"where": where_cand, "timezone": timezone, "delimiter": ",", "with_bom": "false"},
        timeout=300,
    )
    r.raise_for_status()
    sites = pd.read_csv(StringIO(r.text), dtype={"id_site": "string"}, low_memory=False)

    lat = lon = None
    if "coordonnees_geo.lat" in sites.columns and "coordonnees_geo.lon" in sites.columns:
        lat = pd.to_numeric(sites["coordonnees_geo.lat"], errors="coerce")
        lon = pd.to_numeric(sites["coordonnees_geo.lon"], errors="coerce")
    elif "geo_point_2d" in sites.columns:
        tmp = sites["geo_point_2d"].astype(str)
        parts = tmp.str.split(",", expand=True)
        if parts.shape[1] >= 2:
            lat = pd.to_numeric(parts[0].str.strip(), errors="coerce")
            lon = pd.to_numeric(parts[1].str.strip(), errors="coerce")
    elif "coordonnees_geo" in sites.columns:
        tmp = sites["coordonnees_geo"]

        def _parse(x):
            if pd.isna(x):
                return (np.nan, np.nan)
            if isinstance(x, dict):
                return (x.get("lat", np.nan), x.get("lon", np.nan))
            s = str(x).strip()
            try:
                if "{" in s and "lat" in s and "lon" in s:
                    d = ast.literal_eval(s)
                    if isinstance(d, dict):
                        return (d.get("lat", np.nan), d.get("lon", np.nan))
            except Exception:
                pass
            if "," in s:
                p = [z.strip() for z in s.split(",")]
                try:
                    return (float(p[0]), float(p[1]))
                except Exception:
                    return (np.nan, np.nan)
            return (np.nan, np.nan)

        ll = tmp.apply(_parse)
        lat = ll.apply(lambda z: z[0])
        lon = ll.apply(lambda z: z[1])
    elif "lat" in sites.columns and "lon" in sites.columns:
        lat = pd.to_numeric(sites["lat"], errors="coerce")
        lon = pd.to_numeric(sites["lon"], errors="coerce")

    sites2 = pd.DataFrame({
        "id_site": sites["id_site"].astype("string"),
        "label": sites["label"] if "label" in sites.columns else np.nan,
        "lat": lat if lat is not None else np.nan,
        "lon": lon if lon is not None else np.nan,
    })
    sites2 = (
        sites2.dropna(subset=["lat", "lon"])
        .drop_duplicates(subset=["id_site"])
        .reset_index(drop=True)
    )
    if sites2.empty:
        raise RuntimeError(f"sites Le jeu de données n'a résolu aucune coordonnée, print sites.columns:{list(sites.columns)}")

    # Sites candidats triés par heures_présentes
    cand_rank = top_sites_df.copy()
    cand_rank["id_site"] = cand_rank["id_site"].astype(str)
    cand_rank = cand_rank.merge(sites2[["id_site", "lat", "lon"]], on="id_site", how="inner")
    cand_rank = cand_rank.sort_values("hours_present", ascending=False)

    final_sites = cand_rank["id_site"].head(n_sites).tolist()
    if not final_sites:
        raise RuntimeError("Aucun site candidat ne répond simultanément à données + coordonnées")

    print("Le site finalement sélectionné")
    print(final_sites)
    where_sites = "id_site in (" + ",".join([f"'{sid}'" for sid in final_sites]) + ")"

    # Téléchargements mensuels
    t0 = pd.Timestamp(f"{year_start}-01-01", tz="UTC")
    t1 = pd.Timestamp(f"{year_end+1}-01-01", tz="UTC")
    boundaries = list(pd.date_range(t0, t1, freq=chunk_freq, inclusive="left")) + [t1]

    parts = []
    for i in range(len(boundaries) - 1):
        a = boundaries[i].strftime("%Y-%m-%dT%H:%M:%SZ")
        b = boundaries[i + 1].strftime("%Y-%m-%dT%H:%M:%SZ")
        where_chunk = f"t >= '{a}' AND t < '{b}' AND {where_modes} AND {where_sites}"

        r = requests.get(
            f"{base_counts}/exports/csv",
            params={"where": where_chunk, "timezone": timezone, "delimiter": ",", "with_bom": "false"},
            timeout=300,
        )
        r.raise_for_status()
        df = pd.read_csv(
            StringIO(r.text),
            dtype={"id_site": "string", "id_trajectoire": "string"},
            low_memory=False,
        )
        if not df.empty:
            parts.append(df)

    if not parts:
        raise RuntimeError("Les détails du téléchargement sont vides")

    counts = pd.concat(parts, ignore_index=True)
    counts = counts[counts["mode"].isin(modes_keep)].copy()

    # Durée et valeurs du nettoyage
    counts["t_utc"] = pd.to_datetime(counts["t"], errors="coerce", utc=True)
    counts = counts.dropna(subset=["t_utc"])

    counts["nb_usagers"] = pd.to_numeric(counts["nb_usagers"], errors="coerce")
    counts = counts.dropna(subset=["nb_usagers"])

    if "sens" not in counts.columns:
        counts["sens"] = "NA"
    else:
        counts["sens"] = counts["sens"].fillna("NA").astype(str)

    # Agrégé à l'heure
    group_cols = ["id_site", "sens", "mode", "t_utc"]
    if keep_trajectoire and "id_trajectoire" in counts.columns:
        group_cols = ["id_site", "id_trajectoire", "sens", "mode", "t_utc"]

    agg = counts.groupby(group_cols, dropna=False)["nb_usagers"].sum().reset_index()
    idx_cols = [c for c in group_cols if c != "mode"]
    wide = agg.pivot_table(index=idx_cols, columns="mode", values="nb_usagers", aggfunc="sum").reset_index()
    wide.columns.name = None
    # Fusionner les coordonnées du site
    site_coords = sites2[sites2["id_site"].isin(final_sites)].drop_duplicates("id_site")
    wide = wide.merge(site_coords, on="id_site", how="left").dropna(subset=["lat", "lon"])
    sc = wide[["id_site", "lat", "lon"]].drop_duplicates("id_site").copy()
    sc["id_site"] = sc["id_site"].astype(str)
    sc["lat"] = sc["lat"].astype(float)
    sc["lon"] = sc["lon"].astype(float)
    
    lat_str = ",".join(sc["lat"].map(lambda x: f"{x:.6f}").tolist())
    lon_str = ",".join(sc["lon"].map(lambda x: f"{x:.6f}").tolist())
    if wide.empty:
        raise RuntimeError("wide est vide")

    # Plage horaire météo
    tmin = wide["t_utc"].min()
    tmax = wide["t_utc"].max()
    if pd.isna(tmin) or pd.isna(tmax):
        raise RuntimeError("tmin/tmax pour NaT")

    start_date = tmin.strftime("%Y-%m-%d")
    end_date = tmax.strftime("%Y-%m-%d")

    # Regroupez par année (ou semestre), en notant que la date de fin inclut le jour même
    ws = pd.Timestamp(start_date)  # date
    we = pd.Timestamp(end_date)    # date

    # Limite : ws, 1er janvier de l'année suivante, ... , we+1 jour
    boundaries = list(pd.date_range(ws, we + pd.Timedelta(days=1), freq=weather_chunk, inclusive="left"))
    if boundaries[0] != ws:
        boundaries = [ws] + boundaries
    if boundaries[-1] != we + pd.Timedelta(days=1):
        boundaries.append(we + pd.Timedelta(days=1))

    weather_parts = []
    for i in range(len(boundaries) - 1):
        a = boundaries[i]
        b = boundaries[i + 1] - pd.Timedelta(days=1)
        if b < a:
            continue
        if b > we:
            b = we

        a_str = a.strftime("%Y-%m-%d")
        b_str = b.strftime("%Y-%m-%d")

        params = {
            "latitude": lat_str,
            "longitude": lon_str,
            "start_date": a_str,
            "end_date": b_str,
            "hourly": ",".join(weather_vars),
            "timezone": weather_timezone,
        }

        wait = 1.0
        last_err = None
        for attempt in range(weather_max_retries):
            rr = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=180)
            if rr.status_code == 429:
                ra = rr.headers.get("Retry-After")
                if ra is not None:
                    try:
                        wait = max(wait, float(ra))
                    except Exception:
                        pass
                time.sleep(wait)
                wait = min(wait * 2, 60)
                last_err = f"429 on weather chunk {a_str}~{b_str}"
                continue
            if rr.status_code >= 400:
                rr.raise_for_status()

            js = rr.json()
            data_list = js if isinstance(js, list) else [js]

            for loc_idx, item in enumerate(data_list):
                hourly = item.get("hourly", {})
                times = hourly.get("time")
                if not times:
                    continue
                w = pd.DataFrame({"t_utc": pd.to_datetime(times, utc=True)})
                for v in weather_vars:
                    w[v] = hourly.get(v)
                w["id_site"] = sc.iloc[loc_idx]["id_site"] if loc_idx < len(sc) else None
                weather_parts.append(w)

            last_err = None
            break

        if last_err is not None:
            raise RuntimeError(last_err)
        time.sleep(0.5)

    if not weather_parts:
        raise RuntimeError("Les données météorologiques sont vides")

    weather = pd.concat(weather_parts, ignore_index=True)
    weather = weather.dropna(subset=["id_site"])
    weather = pd.concat(weather_parts, ignore_index=True)
    weather = weather.dropna(subset=["id_site"])
    weather = weather.drop_duplicates(["id_site", "t_utc"])
    # Fusion météo
    df_final = wide.merge(weather, on=["id_site", "t_utc"], how="left")

    # Caractéristiques temporelles
    df_final["t_paris"] = df_final["t_utc"].dt.tz_convert("Europe/Paris")
    df_final["hour"] = df_final["t_paris"].dt.hour
    df_final["dow"] = df_final["t_paris"].dt.dayofweek
    df_final["month"] = df_final["t_paris"].dt.month
    df_final["is_weekend"] = (df_final["dow"] >= 5).astype(int)

    df_final["hour_sin"] = np.sin(2 * np.pi * df_final["hour"] / 24.0)
    df_final["hour_cos"] = np.cos(2 * np.pi * df_final["hour"] / 24.0)
    df_final["month_sin"] = np.sin(2 * np.pi * df_final["month"] / 12.0)
    df_final["month_cos"] = np.cos(2 * np.pi * df_final["month"] / 12.0)

    if out_parquet:
        df_final.to_parquet(out_parquet, index=False)
        print(f"Saved: {out_parquet}")

    print("shape:", df_final.shape)
    print(df_final.head(10))
    return df_final

In [5]:
df = build_traffic_weather_dataset(year_start=2021, year_end=2026, n_sites=10, keep_trajectoire=False, out_parquet="traffic_weather.parquet")

Le site finalement sélectionné
['10022', '10023', '10004', '10015', '10019', '10020', '10029', '10030', '10028', '10081']


NameError: name 'weather_vars' is not defined

In [ ]:
def tw_quality_and_pick(
    parquet_path="traffic_weather.parquet",
    modes=None,
    target_mode="Véhicules légers < 3,5t",
    min_target_nonnull=0.90, # Seuil du ratio de colonnes cibles non vides
    min_days=365, 
):

    df = pd.read_parquet(parquet_path)

    if modes is None:
        modes = [
            "Trottinettes + vélos",
            "2 roues motorisées",
            "Trottinettes",
            "Véhicules légers < 3,5t",
            "Véhicules lourds > 3,5t",
        ]

    key = ["id_site", "sens", "t_utc"]
    dup = df.duplicated(key).sum()
    print("Nombre de lignes de la répétition (id_site,sens,t_utc):", int(dup))

    # Période
    print("Temps globale:", df["t_utc"].min(), "->", df["t_utc"].max())
    print("Nombre de tous lignes:", len(df), "Nombre de sites:", df["id_site"].nunique(), "Numéro de direction:", df["sens"].nunique())

    # Rapport sur le site et direction
    rows = []
    for (sid, sens), g in df.groupby(["id_site", "sens"]):
        tmin, tmax = g["t_utc"].min(), g["t_utc"].max()
        expected_hours = int(round((tmax - tmin).total_seconds() / 3600.0)) + 1
        hours_present = g["t_utc"].nunique()
        coverage_days = expected_hours / 24.0
        completeness = hours_present / expected_hours if expected_hours > 0 else np.nan

        item = {
            "id_site": sid,
            "sens": sens,
            "tmin": tmin,
            "tmax": tmax,
            "coverage_days": coverage_days,
            "completeness": completeness,
        }
        for m in modes:
            if m in g.columns:
                item[f"nonnull_{m}"] = float(g[m].notna().mean())
            else:
                item[f"nonnull_{m}"] = np.nan
        rows.append(item)

    rep = pd.DataFrame(rows).sort_values(["coverage_days", "completeness"], ascending=[False, False])
    print("\nTop20")
    print(rep.head(20).to_string(index=False))
    
    # Sélectionner la séquence cible
    col = f"nonnull_{target_mode}"
    if col not in rep.columns:
        print(f"\nLa colonne target_mode est introuvable：{target_mode}")
        return rep, pd.DataFrame()

    picked = rep[
        (rep["coverage_days"] >= min_days) &
        (rep[col] >= min_target_nonnull)
    ].copy().sort_values([col, "coverage_days"], ascending=[False, False])

    print(f"\n*{target_mode} id_site×sens peut être utilisé pour la prédiction, target non vide>={min_target_nonnull:.0%}, Couverture>={min_days}jours）===")
    print(picked[["id_site", "sens", "coverage_days", "completeness", col]].head(30).to_string(index=False))

    return rep, picked


In [ ]:
rep, picked = tw_quality_and_pick(
    parquet_path="traffic_weather.parquet",
    target_mode="Véhicules légers < 3,5t",
    min_target_nonnull=0.90,
    min_days=365,
)

In [ ]:
df = pd.read_parquet("traffic_weather.parquet")
df["id_site"] = df["id_site"].astype(str)
df["sens"] = df["sens"].astype(str)
df["t_utc"] = pd.to_datetime(df["t_utc"], utc=True, errors="coerce")
df = df.dropna(subset=["t_utc"])
df = df[df["sens"] != "NA"].copy()

key = ["id_site", "sens", "t_utc"]
traffic_cols = [
    "Trottinettes + vélos",
    "2 roues motorisées",
    "Trottinettes",
    "Véhicules légers < 3,5t",
    "Véhicules lourds > 3,5t",
]
traffic_cols = [c for c in traffic_cols if c in df.columns]
weather_cols = [c for c in ["temperature_2m","relative_humidity_2m","precipitation","rain","snowfall",
                            "surface_pressure","cloud_cover","wind_speed_10m","wind_direction_10m"] if c in df.columns]
other_cols = [c for c in df.columns if c not in set(key + traffic_cols + weather_cols)]

agg = {c: "sum" for c in traffic_cols}
agg.update({c: "mean" for c in weather_cols})
agg.update({c: "first" for c in other_cols})

print("dup before:", int(df.duplicated(key).sum()))
df = df.groupby(key, as_index=False).agg(agg)
print("dup after :", int(df.duplicated(key).sum()))

df["t_paris"] = df["t_utc"].dt.tz_convert("Europe/Paris")
df["hour"] = df["t_paris"].dt.hour
df["dow"] = df["t_paris"].dt.dayofweek
df["month"] = df["t_paris"].dt.month
df["is_weekend"] = (df["dow"] >= 5).astype(int)
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

df.to_parquet("traffic_weather_clean.parquet", index=False)

In [ ]:
def export_mode_datasets_3sites_with_sens(
    parquet_in="traffic_weather.parquet",
    out_dir="mode_datasets_3sites_sens",
    site_ids=("10022", "10023", "10004"),
    sens_keep=("E-O", "O-E"), # deux directions
    modes_keep=(
        "Trottinettes + vélos",
        "2 roues motorisées",
        "Trottinettes",
        "Véhicules légers < 3,5t",
        "Véhicules lourds > 3,5t",
    ),
    dedup=False,
    time_intersection=True,
):

    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_parquet(parquet_in)

    df["id_site"] = df["id_site"].astype(str)
    df["sens"] = df["sens"].astype(str)
    df["t_utc"] = pd.to_datetime(df["t_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["t_utc"])

    df = df[df["id_site"].isin([str(x) for x in site_ids])].copy()
    df = df[df["sens"].isin(list(sens_keep))].copy()

    key = ["id_site", "sens", "t_utc"]

    if dedup:
        traffic_cols = [c for c in modes_keep if c in df.columns]
        weather_set = {
            "temperature_2m","relative_humidity_2m","precipitation","rain","snowfall",
            "surface_pressure","cloud_cover","wind_speed_10m","wind_direction_10m"
        }
        weather_cols = [c for c in df.columns if c in weather_set]
        other_cols = [c for c in df.columns if c not in set(key + traffic_cols + weather_cols)]

        agg = {c: "sum" for c in traffic_cols}
        agg.update({c: "mean" for c in weather_cols})
        agg.update({c: "first" for c in other_cols})

        df = df.groupby(key, as_index=False).agg(agg)

    if time_intersection:
        g = df.groupby(["id_site","sens"])["t_utc"]
        t_start = g.min().max()
        t_end = g.max().min()
        df = df[(df["t_utc"] >= t_start) & (df["t_utc"] <= t_end)].copy()

    weather_cols = [c for c in [
        "temperature_2m","relative_humidity_2m","precipitation","rain","snowfall",
        "surface_pressure","cloud_cover","wind_speed_10m","wind_direction_10m"
    ] if c in df.columns]
    time_cols = [c for c in ["hour","dow","month","is_weekend","hour_sin","hour_cos","month_sin","month_cos"] if c in df.columns]
    meta_cols = [c for c in ["label","lat","lon"] if c in df.columns]

    written = []
    for m in modes_keep:
        if m not in df.columns:
            continue

        out = df[key + [m] + weather_cols + time_cols + meta_cols].copy()
        safe = (
            m.replace(" ", "_")
             .replace("+", "plus")
             .replace("<", "lt")
             .replace(">", "gt")
             .replace(",", "")
             .replace("é", "e").replace("è", "e").replace("ê", "e").replace("à", "a")
        )
        path = os.path.join(out_dir, f"mode__{safe}.parquet")
        out.to_parquet(path, index=False)
        written.append((m, path, out.shape))
    for x in written:
        print(x)
    return written


In [ ]:
export_mode_datasets_3sites_with_sens()

In [ ]:
import glob
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
def train_models_sens(
    data_dir="mode_datasets_3sites_sens",
    model_dir="models_3sites_sens",
    n_lags=24,
    rolling_windows=(24,),
    test_ratio=0.15,
    val_ratio=0.15,
    skip_if_test_nonzero_below=0.01, # Si c'est trop clairsemé, passez votre chemin.
):

    os.makedirs(model_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(data_dir, "mode__*.parquet")))
    if not files:
        raise RuntimeError("Parquet de données d'entraînement introuvable")
    metrics = []

    for fp in files:
        df = pd.read_parquet(fp)

        if "label" in df.columns:
            df = df.drop(columns=["label"])

        df["id_site"] = df["id_site"].astype(str)
        df["sens"] = df["sens"].astype(str)
        df["t_utc"] = pd.to_datetime(df["t_utc"], utc=True, errors="coerce")
        df = df.dropna(subset=["t_utc"]).sort_values(["id_site","sens","t_utc"]).reset_index(drop=True)

        exclude = {
            "id_site","sens","t_utc","lat","lon",
            "hour","dow","month","is_weekend","hour_sin","hour_cos","month_sin","month_cos",
            "temperature_2m","relative_humidity_2m","precipitation","rain","snowfall","surface_pressure",
            "cloud_cover","wind_speed_10m","wind_direction_10m",
        }
        target_candidates = [c for c in df.columns if c not in exclude]
        if len(target_candidates) != 1:
            raise RuntimeError(f"Peut pas identifier le target：{fp} -> {target_candidates}")
        target = target_candidates[0]

        # Tout d'abord, convertissez chaque paire (id_site, sens) en une grille horaire stricte
        parts = []
        for (sid, sens), g in df.groupby(["id_site","sens"], sort=False):
            g = g.sort_values("t_utc")
            rng = pd.date_range(g["t_utc"].min(), g["t_utc"].max(), freq="H", tz="UTC")
            g2 = g.set_index("t_utc").reindex(rng)
            g2["t_utc"] = g2.index
            g2["id_site"] = sid
            g2["sens"] = sens
            parts.append(g2.reset_index(drop=True))
        df = pd.concat(parts, ignore_index=True)
        # Recalculer les caractéristiques temporelles
        df["t_paris"] = df["t_utc"].dt.tz_convert("Europe/Paris")
        df["hour"] = df["t_paris"].dt.hour
        df["dow"] = df["t_paris"].dt.dayofweek
        df["month"] = df["t_paris"].dt.month
        df["is_weekend"] = (df["dow"] >= 5).astype(int)
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12.0)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12.0)

        # y：les valeurs manquantes sont conservées sous forme de NaN.
        y_raw = pd.to_numeric(df[target], errors="coerce")
        y_raw = y_raw.clip(lower=0)
        df["y"] = np.where(y_raw.isna(), np.nan, np.log1p(y_raw))

        gkey = ["id_site","sens"]

        # lags / rolling 
        for k in range(1, n_lags + 1):
            df[f"y_lag_{k}"] = df.groupby(gkey)["y"].shift(k)

        for w in rolling_windows:
            s = (
                df.groupby(gkey)["y"]
                  .apply(lambda x: x.shift(1).rolling(w, min_periods=w).mean())
            )
            df[f"y_rollmean_{w}"] = s.reset_index(level=gkey, drop=True)

        # Conserver uniquement la cible existe et données historiques suffisantes
        df2 = df.dropna(subset=["y", f"y_lag_{n_lags}"]).copy()

        # Répartition dans le temps
        ts = np.sort(df2["t_utc"].unique())
        n = len(ts)
        n_test = int(round(n * test_ratio))
        n_val = int(round(n * val_ratio))
        n_train = n - n_test - n_val
        if n_train <= 0:
            raise RuntimeError("Le ratio de répartition du temps est déraisonnable.")

        t_train_end = ts[n_train - 1]
        t_val_end = ts[n_train + n_val - 1]

        train = df2[df2["t_utc"] <= t_train_end]
        val = df2[(df2["t_utc"] > t_train_end) & (df2["t_utc"] <= t_val_end)]
        test = df2[df2["t_utc"] > t_val_end]

        # Le mode clairsemé passe directement
        test_nonzero = float((np.expm1(test["y"]) > 0).mean())
        if test_nonzero < skip_if_test_nonzero_below:
            print(f"[{target}] skip: test_nonzero={test_nonzero:.3f}")
            continue
        # Chronique
        lag_cols = [f"y_lag_{k}" for k in range(1, n_lags + 1)]
        roll_cols = [f"y_rollmean_{w}" for w in rolling_windows]
        num_cols = (
            lag_cols + roll_cols +
            ["lat","lon","hour","dow","month","is_weekend","hour_sin","hour_cos","month_sin","month_cos",
             "temperature_2m","relative_humidity_2m","precipitation","rain","snowfall","surface_pressure",
             "cloud_cover","wind_speed_10m","wind_direction_10m"]
        )
        num_cols = [c for c in num_cols if c in df2.columns]
        cat_cols = ["id_site","sens"]

        X_train, y_train = train[num_cols + cat_cols], train["y"]
        X_val, y_val = val[num_cols + cat_cols], val["y"]
        X_test, y_test = test[num_cols + cat_cols], test["y"]

        pre = ColumnTransformer(
            transformers=[
                ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ],
            remainder="drop",
        )

        model = HistGradientBoostingRegressor(
            max_depth=None,
            learning_rate=0.05,
            max_iter=300,
            random_state=0,
        )

        pipe = Pipeline([("pre", pre), ("model", model)])
        pipe.fit(X_train, y_train)

        pred = pipe.predict(X_test)

        rmse = mean_squared_error(y_test, pred, squared=False)
        mae = mean_absolute_error(y_test, pred)
        r2 = r2_score(y_test, pred)

        out_path = os.path.join(model_dir, f"hgb__{os.path.basename(fp).replace('.parquet','.joblib')}")
        joblib.dump({"model": pipe, "target": target, "num_cols": num_cols, "cat_cols": cat_cols}, out_path)

        print(f"[{target}] rmse={rmse:.4f} mae={mae:.4f} r2={r2:.4f} saved={out_path}")

        metrics.append({"target": target, "rmse": rmse, "mae": mae, "r2": r2, "test_nonzero": test_nonzero})

    if metrics:
        mdf = pd.DataFrame(metrics).sort_values("rmse")
        print("\n=== metrics ===")
        print(mdf)
        return mdf
    return None

In [ ]:
train_models_sens()

In [ ]:
import json, re, random
import torch
import torch.nn as nn

In [ ]:
def train_gru_all_modes(
    data_dir="mode_datasets_3sites_sens",
    out_dir="models_nn",
    recent_days=730, # 2 ans
    lookback=72, # période de 3 jours
    epochs=15,
    batch_size=1024,
    lr=1e-3,
    patience=3,
    test_ratio=0.15,
    val_ratio=0.15,
    hidden=128,
    num_layers=2,
    dropout=0.2,
    weight_decay=1e-4,
    clip_grad=1.0,
    pos_weight=3.0, # Base pondérée non nulle 
    value_weight_alpha=0.25, # Pondération en fonction du volume de trafic
    weight_cap=10.0, # Limite de pondération
    mask_missing_target=True,
    max_train_samples_per_epoch=200000,
    max_val_samples=80000,
    max_test_samples=None,
    # le test ignore les cas où la proportion non nulle est inférieure au seuil
    skip_if_test_nonzero_below=0.01,
    seed=0,
):

    os.makedirs(out_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(data_dir, "mode__*.parquet")))
    if not files:
        raise RuntimeError(f"Introuvable {data_dir}/mode__*.parquet")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

    device = "cuda" if torch.cuda.is_available() else "cpu"

    weather_candidates = [
        "temperature_2m","relative_humidity_2m","precipitation","wind_speed_10m",
        "rain","snowfall","surface_pressure","cloud_cover","wind_direction_10m",
    ]

    results = []

    for fp in files:
        df = pd.read_parquet(fp)
        if "label" in df.columns:
            df = df.drop(columns=["label"])

        df["id_site"] = df["id_site"].astype(str)
        df["sens"] = df["sens"].astype(str)
        df["t_utc"] = pd.to_datetime(df["t_utc"], utc=True, errors="coerce")
        df = df.dropna(subset=["t_utc"]).sort_values(["id_site","sens","t_utc"]).reset_index(drop=True)

        # des deux dernières années
        if recent_days is not None and recent_days > 0:
            end = df["t_utc"].max()
            start = end - pd.Timedelta(days=int(recent_days))
            df = df[(df["t_utc"] >= start) & (df["t_utc"] <= end)].copy()
            df = df.reset_index(drop=True)

        exclude = {
            "id_site","sens","t_utc","lat","lon",
            "hour","dow","month","is_weekend","hour_sin","hour_cos","month_sin","month_cos",
            *weather_candidates
        }
        target_candidates = [c for c in df.columns if c not in exclude]
        if len(target_candidates) != 1:
            raise RuntimeError(f"Impossible de reconnaître la cible：{os.path.basename(fp)} -> {target_candidates}")
        target = target_candidates[0]

        if only_targets is not None and target not in set(only_targets):
            continue

        weather_cols = [c for c in weather_candidates if c in df.columns]

        site_list = sorted(df["id_site"].unique().tolist())
        sens_list = sorted(df["sens"].unique().tolist())
        site2i = {s:i for i,s in enumerate(site_list)}
        sens2i = {s:i for i,s in enumerate(sens_list)}
        n_site = len(site_list)
        n_sens = len(sens_list)

        series = []
        for (sid, sens), g in df.groupby(["id_site","sens"]):
            g = g.sort_values("t_utc")
            idx = pd.date_range(g["t_utc"].min(), g["t_utc"].max(), freq="h", tz="UTC")
            g = g.set_index("t_utc").reindex(idx)

            y = pd.to_numeric(g[target], errors="coerce")
            miss = y.isna().astype(np.float32).values
            y_raw = y.fillna(0.0).astype(np.float32).values
            y_log = np.log1p(y_raw).astype(np.float32)

            W = []
            for c in weather_cols:
                w = pd.to_numeric(g[c], errors="coerce")
                w = w.ffill().bfill().fillna(0.0).astype(np.float32).values
                W.append(w)
            W = np.stack(W, axis=1) if W else np.zeros((len(idx), 0), dtype=np.float32)

            t_paris = pd.DatetimeIndex(idx).tz_convert("Europe/Paris")
            hour = t_paris.hour.values.astype(np.float32)
            dow = t_paris.dayofweek.values.astype(np.float32)
            month = t_paris.month.values.astype(np.float32)
            is_weekend = (dow >= 5).astype(np.float32)

            hour_sin = np.sin(2*np.pi*hour/24.0).astype(np.float32)
            hour_cos = np.cos(2*np.pi*hour/24.0).astype(np.float32)
            month_sin = np.sin(2*np.pi*month/12.0).astype(np.float32)
            month_cos = np.cos(2*np.pi*month/12.0).astype(np.float32)
            # intégration de cycles hebdomadaires
            dow_sin = np.sin(2*np.pi*dow/7.0).astype(np.float32)
            dow_cos = np.cos(2*np.pi*dow/7.0).astype(np.float32)

            Tfeat = np.stack([hour_sin, hour_cos, month_sin, month_cos, dow_sin, dow_cos, is_weekend], axis=1).astype(np.float32)

            svec = np.zeros((n_site,), dtype=np.float32); svec[site2i[sid]] = 1.0
            dvec = np.zeros((n_sens,), dtype=np.float32); dvec[sens2i[sens]] = 1.0
            const = np.concatenate([svec, dvec]).astype(np.float32)
            C = np.repeat(const[None, :], len(idx), axis=0)

            feat = np.concatenate([y_log[:, None], miss[:, None], W, Tfeat, C], axis=1).astype(np.float32)

            series.append({
                "sid": sid, "sens": sens,
                "idx": idx,
                "feat": feat,
                "y_log": y_log,
                "y_raw": y_raw,
                "miss": miss,
            })

        all_times = np.unique(np.concatenate([s["idx"].values for s in series]))
        all_times.sort()
        n = len(all_times)
        n_test = int(round(n * test_ratio))
        n_val = int(round(n * val_ratio))
        n_train = n - n_test - n_val
        if n_train <= 0:
            raise RuntimeError("Le ratio de répartition du temps est déraisonnable")

        t_train_end = all_times[n_train - 1]
        t_val_end = all_times[n_train + n_val - 1]

        train_idx, val_idx, test_idx = [], [], []
        for si, s in enumerate(series):
            idx = s["idx"].values
            T = len(idx)
            for i in range(lookback, T):
                if mask_missing_target and s["miss"][i] == 1.0:
                    continue
                t = idx[i]
                if t <= t_train_end:
                    train_idx.append((si, i))
                elif t <= t_val_end:
                    val_idx.append((si, i))
                else:
                    test_idx.append((si, i))

        if len(train_idx) < 1000 or len(val_idx) < 200 or len(test_idx) < 200:
            raise RuntimeError(f"moins de l'échantillon：train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

        def nz_stats(idx_list):
            ys = np.array([series[si]["y_raw"][i] for (si, i) in idx_list], dtype=np.float32)
            return float((ys > 0).mean()), float(ys.mean()), float(np.quantile(ys, 0.95))

        nz_tr = nz_stats(train_idx)
        nz_te = nz_stats(test_idx)
        print(f"\n[{target}] recent_days={recent_days} | test nonzero={nz_te[0]:.3f} mean={nz_te[1]:.2f} p95={nz_te[2]:.2f}")

        # Ignorer le mode test avec tous 0 ou presque tous les 0, ex: Trottinettes+vélos
        if nz_te[0] < skip_if_test_nonzero_below:
            print(f"[{target}] test Presque tous les 0, ignorer ce mode")
            continue

        F = series[0]["feat"].shape[1]
        onehot_dim = n_site + n_sens
        onehot_start = F - onehot_dim

        step = max(1, len(train_idx)//50000)
        feats_train = np.stack([series[si]["feat"][i-1, :onehot_start] for si, i in train_idx[::step]], axis=0)
        mu = feats_train.mean(axis=0).astype(np.float32)
        sd = (feats_train.std(axis=0) + 1e-6).astype(np.float32)
        for s in series:
            x = s["feat"]
            x[:, :onehot_start] = (x[:, :onehot_start] - mu) / sd
            s["feat"] = x

        class GRU1(nn.Module):
            def __init__(self, in_dim, hidden=128, num_layers=2, dropout=0.2):
                super().__init__()
                self.gru = nn.GRU(
                    input_size=in_dim,
                    hidden_size=hidden,
                    num_layers=num_layers,
                    dropout=dropout if num_layers > 1 else 0.0,
                    batch_first=True,
                )
                self.fc = nn.Linear(hidden, 1)
            def forward(self, x):
                out, _ = self.gru(x)
                return self.fc(out[:, -1, :]).squeeze(1)
        model = GRU1(F, hidden=hidden, num_layers=num_layers, dropout=dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        huber = nn.SmoothL1Loss(reduction="none")

        def cap_list(idx_list, cap):
            if cap is None or len(idx_list) <= cap:
                return idx_list
            sel = np.random.choice(len(idx_list), size=int(cap), replace=False)
            return [idx_list[j] for j in sel]

        # Mettre automatiquement à l'échelle pos_weight
        nz_tr_ratio = nz_tr[0]
        pos_w_eff = float(pos_weight * (1.0 - nz_tr_ratio) / (nz_tr_ratio + 1e-6))

        def run_epoch(idx_list, train_mode):
            if train_mode:
                idx_list = cap_list(idx_list, max_train_samples_per_epoch)
                model.train()
                random.shuffle(idx_list)
            else:
                idx_list = cap_list(idx_list, max_val_samples)
                model.eval()

            total = 0.0
            nobs = 0

            for b0 in range(0, len(idx_list), batch_size):
                batch = idx_list[b0:b0+batch_size]
                X = np.stack([series[si]["feat"][i-lookback:i] for (si, i) in batch], axis=0).astype(np.float32)
                y = np.stack([series[si]["y_log"][i] for (si, i) in batch], axis=0).astype(np.float32)
                y_raw = np.stack([series[si]["y_raw"][i] for (si, i) in batch], axis=0).astype(np.float32)

                X = torch.from_numpy(X).to(device)
                y = torch.from_numpy(y).to(device)
                # Pondération, non nul + volume de trafic
                w = 1.0 + pos_w_eff * (y_raw > 0).astype(np.float32) + value_weight_alpha * np.log1p(y_raw)
                w = np.minimum(w, weight_cap).astype(np.float32)
                w = torch.from_numpy(w).to(device)

                if train_mode:
                    opt.zero_grad()
                    pred = model(X)
                    loss = (huber(pred, y) * w).mean()
                    loss.backward()
                    if clip_grad and clip_grad > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                    opt.step()
                else:
                    with torch.no_grad():
                        pred = model(X)
                        loss = (huber(pred, y) * w).mean()

                total += float(loss.item()) * len(batch)
                nobs += len(batch)
            return total / max(1, nobs)

        best = 1e18
        bad = 0
        safe = re.sub(r"[^A-Za-z0-9_]+", "_", target)
        best_path = os.path.join(out_dir, f"tmp_best__{safe}.pt")

        for ep in range(1, epochs+1):
            tr = run_epoch(train_idx, True)
            va = run_epoch(val_idx, False)
            print(f"{target} | ep {ep}/{epochs} | train {tr:.4f} | val {va:.4f}")
            if va + 1e-6 < best:
                best = va
                bad = 0
                torch.save(model.state_dict(), best_path)
            else:
                bad += 1
                if bad >= patience:
                    break

        model.load_state_dict(torch.load(best_path, map_location=device))
        model.eval()

        te_list = cap_list(test_idx, max_test_samples)

        preds = []
        trues = []
        with torch.no_grad():
            for b0 in range(0, len(te_list), batch_size):
                batch = te_list[b0:b0+batch_size]
                X = np.stack([series[si]["feat"][i-lookback:i] for (si, i) in batch], axis=0).astype(np.float32)
                y = np.stack([series[si]["y_log"][i] for (si, i) in batch], axis=0).astype(np.float32)
                p = model(torch.from_numpy(X).to(device)).cpu().numpy()
                preds.append(p); trues.append(y)

        p = np.concatenate(preds)
        y = np.concatenate(trues)
        p_raw = np.expm1(p)
        y_raw = np.expm1(y)
        mae = float(np.mean(np.abs(y_raw - p_raw)))
        rmse = float(np.sqrt(np.mean((y_raw - p_raw)**2)))
        w_path = os.path.join(out_dir, f"{safe}.pt")
        meta_path = os.path.join(out_dir, f"{safe}.json")

        torch.save(model.state_dict(), w_path)
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump({
                "target": target,
                "recent_days": recent_days,
                "lookback": lookback,
                "feature_dim": int(F),
                "weather_cols": weather_cols,
                "time_feats": ["hour_sin","hour_cos","month_sin","month_cos","dow_sin","dow_cos","is_weekend"],
                "site_list": site_list,
                "sens_list": sens_list,
                "mu": mu.tolist(),
                "sd": sd.tolist(),
                "onehot_dim": int(onehot_dim),
                "hidden": int(hidden),
                "num_layers": int(num_layers),
                "dropout": float(dropout),
                "pos_weight_base": float(pos_weight),
                "pos_weight_effective": float(pos_w_eff),
                "value_weight_alpha": float(value_weight_alpha),
                "mask_missing_target": bool(mask_missing_target),
            }, f, ensure_ascii=False, indent=2)

        print(f"sauvegarter NN: {w_path} | teste MAE={mae:.3f} RMSE={rmse:.3f} (counts)")
        results.append((target, mae, rmse, w_path, meta_path))

    res = pd.DataFrame(results, columns=["target","mae","rmse","weights_path","meta_path"]).sort_values("rmse")
    res.to_csv(os.path.join(out_dir, "metrics.csv"), index=False)
    print(res)
    return res

In [ ]:
train_gru_all_modes()